# Fine-tune Phi-3-mini on MedQuAD with LoRA

**UF AI Passport — Project 7 capstone.** QLoRA fine-tuning of `microsoft/Phi-3-mini-4k-instruct` on MedQuAD (public NIH Q&A, CC BY 4.0), then merge and export to GGUF for the local Ollama app.

Only public data is used here. The app itself, along with your documents and questions, never touches the cloud.

**Where to run.** The notebook detects its environment:
- **UF HiPerGator** (Open OnDemand Jupyter, e.g. one L4 GPU): follow [`training/hpc/README.md`](hpc/README.md). Open this notebook from the repo's `training/` folder with the **PHA training** kernel.
- **Google Colab:** choose *Runtime → Change runtime type → GPU*, and upload `prepare_medquad.py`, `lora_pipeline.py`, `system_prompt.txt` and `export_gguf.sh` to `MyDrive/pha/`.

Keep `SMOKE_TEST = True` the first time (about 5–10 minutes), then set it to `False` for the full run. If the session ends, re-run all cells: training resumes from the last checkpoint.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # on HiPerGator these are preinstalled by training/hpc/setup_env.sh
    %pip install -q transformers==5.18.0 peft==0.21.2 trl==1.14.1 accelerate==1.15.0 datasets==5.0.1 bitsandbytes==0.50.2

In [ ]:
import os
from pathlib import Path

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/pha")
    OUT = WORK
else:  # Jupyter, e.g. HiPerGator: run from the repo's training/ folder
    WORK = Path(os.environ.get("PHA_WORK", Path.cwd())).resolve()
    OUT = Path(os.environ.get("PHA_OUTPUT", WORK / "runs")).resolve()  # big files: use /blue
needed = [
    "prepare_medquad.py",
    "prompt_format.py",
    "synthetic_records.py",
    "lora_pipeline.py",
    "system_prompt.txt",
    "rag_instructions.txt",
    "no_context_instructions.txt",
    "export_gguf.sh",
]
missing = [f for f in needed if not (WORK / f).is_file()]
assert not missing, (
    f"{WORK} is missing {missing}. Open the notebook from training/ or set PHA_WORK."
)
sys.path.insert(0, str(WORK))
OUT.mkdir(parents=True, exist_ok=True)
# VERSION 3 = v2 + multi-turn follow-ups, the app's extraction format, no-range tests, absent
#             medications, Key Points summaries removed (openspec: add-grounded-finetune-v3).
# VERSION 2 = grounded fine-tune (cleaned MedQuAD in the app's RAG format + synthetic records).
# VERSION 1 = the original closed-book run.
VERSION = 3
V2 = VERSION >= 2  # grounded data format (v2 and v3)
NAME = {1: "phi3-medquad", 2: "phi3-medquad-v2", 3: "phi3-medquad-v3"}[VERSION]
DATA = (
    OUT / "data" / {1: "medquad", 2: "medquad-v2", 3: "medquad-v3"}[VERSION]
)  # skipped cells still work

import torch

assert torch.cuda.is_available(), "No GPU visible. Request a GPU for this Jupyter session."
props = torch.cuda.get_device_properties(0)
print(
    f"GPU: {props.name}, {props.total_memory / 2**30:.0f} GB, bf16={torch.cuda.is_bf16_supported()}"
)
print(f"WORK={WORK}\nOUT={OUT}\nHF_HOME={os.environ.get('HF_HOME', '~/.cache/huggingface')}")

## 1. Prepare MedQuAD
This downloads the pinned MedQuAD commit and writes leak-free train/val/test splits, grouped by medical topic.

With `VERSION = 2` or `3` it also removes dataset boilerplate and renders examples in the app's retrieval format: closed-book, answer-plus-distractor excerpts, and distractor-only. It also adds synthetic personal-record questions: 2,500 for v2; 4,000 for v3, more than half of them multi-turn follow-ups. See `openspec/changes/archive/2026-10-06-add-grounded-finetune-v2` and `openspec/changes/add-grounded-finetune-v3`.

In [ ]:
V2_FLAGS = {
    1: "",
    2: "--clean-templates --grounded --synthetic-train 2500 --synthetic-val 250",
    3: "--clean-templates --grounded --v3 --synthetic-train 4000 --synthetic-val 400",
}[VERSION]
!python {WORK}/prepare_medquad.py --download --out {DATA} --system-prompt {WORK}/system_prompt.txt {V2_FLAGS}

## 2. Configure
All hyperparameters are in one place. They are saved next to the adapter as `training_config.json`.

In [ ]:
import lora_pipeline as lp

SMOKE_TEST = True  # first run: ~50 steps to check the setup. Then set False for the full run.

cfg = lp.TrainConfig(
    base_model="microsoft/Phi-3-mini-4k-instruct",
    data_dir=str(DATA),
    output_dir=str(OUT / "outputs" / (f"{NAME}-smoke" if SMOKE_TEST else NAME)),
    seed=13,
    max_length=2048 if V2 else 1024,  # v2 prompts carry up to 4 retrieved excerpts
    load_in_4bit=not V2,  # v1: QLoRA (NF4); v2/v3: LoRA on a bf16 base (fits 2x L4)
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup=0.03,
    num_train_epochs=1.0,  # try 2 if validation loss is still falling at the end
    # v2/v3 as recorded in results/v2/training_config.json: 2 per GPU; the pipeline divides the
    # accumulation by the number of GPUs, so 2 x (8 / 2) x 2 GPUs = effective batch 16.
    per_device_train_batch_size=2 if V2 else 4,
    gradient_accumulation_steps=8 if V2 else 4,
    eval_steps=200,
    save_steps=200,
    logging_steps=20,
)
if SMOKE_TEST:
    cfg.max_steps, cfg.eval_subset = 50, 200
    cfg.eval_steps, cfg.save_steps, cfg.logging_steps = 25, 25, 5
cfg

## 3. Train
This evaluates validation loss every `eval_steps`, keeps the best checkpoint, and resumes automatically after a disconnect.

**Several GPUs** (e.g. 2× L4): training runs *data parallel*. Each GPU holds a full model copy and processes half of every batch, launched with `torchrun` from this cell. Gradient accumulation is divided by the GPU count, so the effective batch stays 16 and the results are comparable to a 1-GPU run. Worker output streams below.

In [ ]:
n_gpus = torch.cuda.device_count()
print(f"Training on {n_gpus} GPU(s)" + (" (data parallel via torchrun)" if n_gpus > 1 else ""))
report = lp.train(cfg) if n_gpus == 1 else lp.train_multi_gpu(cfg, n_gpus)

keys = (
    "best_eval_loss",
    "final_eval_loss",
    "train_loss",
    "global_steps",
    "gpus",
    "effective_batch_size",
    "best_checkpoint",
)
{k: report.get(k) for k in keys} | report["data"]

## 4. Qualitative check: base vs fine-tuned
Ten held-out test questions, answered with the adapter switched on and off. The test split shares no medical topics with training.

*Resuming after training finished?* Run the setup cells and the config cell (with the same `SMOKE_TEST` value), skip **3. Train**, and continue from here.

In [ ]:
import gc
import json
import random

import pandas as pd
from peft import PeftModel

system_prompt = (WORK / "system_prompt.txt").read_text().strip()
test_rows = [json.loads(line) for line in (DATA / "test.jsonl").read_text().splitlines()]
questions = [r["messages"][1]["content"] for r in random.Random(0).sample(test_rows, 10)]

tok = lp.load_tokenizer(str(cfg.adapter_dir))
model = PeftModel.from_pretrained(lp.load_base_model(cfg), str(cfg.adapter_dir))
rows = lp.compare_answers(model, tok, system_prompt, questions, max_new_tokens=256)
(cfg.adapter_dir / "qualitative_samples.json").write_text(json.dumps(rows, indent=2))

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows).style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

## 5. Merge the adapter into an fp16 base
The adapter is merged into full-precision weights, never the 4-bit ones. The original tokenizer files are copied too, because llama.cpp needs `tokenizer.model`.

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()
MERGED = OUT / "outputs" / (f"{NAME}-smoke-merged" if SMOKE_TEST else f"{NAME}-merged")
lp.merge_adapter(cfg.base_model, cfg.adapter_dir, MERGED)
sorted(p.name for p in MERGED.iterdir())

## 6. Export to GGUF (Q4_K_M)
This builds the pinned llama.cpp and converts both models with the same pipeline:
- `NAME` (for example `phi3-medquad-v3`): the fine-tuned model
- `phi3-base`: the unmodified base, for a fair comparison in the evaluation suite

Each file is about 2.3 GB. The export needs `cmake` and a C++ compiler. On HiPerGator, load them first: `module load gcc cmake` (see `hpc/README.md`).

In [ ]:
import shutil

assert shutil.which("cmake") and shutil.which("c++"), (
    "Need cmake + a C++ compiler (HiPerGator: module load gcc cmake)"
)
GGUF = OUT / "gguf"
!bash {WORK}/export_gguf.sh {MERGED} {NAME} {GGUF}
if not SMOKE_TEST and not (GGUF / "phi3-base-Q4_K_M.gguf").exists():
    !bash {WORK}/export_gguf.sh microsoft/Phi-3-mini-4k-instruct phi3-base {GGUF}
!ls -lh {GGUF}

## 7. Bring the models home

Copy both `.gguf` files from `OUT/gguf/` into the repo's `models/` folder on your laptop. On HiPerGator:

```bash
rsync -avP <gatorlink>@hpg.rc.ufl.edu:<OUT>/gguf/*.gguf models/
```

On Colab, download them from `MyDrive/pha/gguf/`. Then:

```bash
cd models
ollama create phi3-medquad-v3 -f Modelfile.medquad-v3   # or the NAME you trained
ollama create phi3-base       -f Modelfile.base          # only if not installed yet
```

Also copy `outputs/<NAME>/adapter/training_config.json` and `qualitative_samples.json` into the repo's `training/results/`, so the model card can quote them.